# STEAM PROJECT visualizations

In [0]:
from pyspark.sql import functions as F, Window

# sources
fact_games = spark.table("steam.fact_games")
bridge_game_publisher = spark.table("steam.bridge_game_publisher")
bridge_game_category = spark.table("steam.bridge_game_category")
bridge_game_genre = spark.table("steam.bridge_game_genre")
bridge_game_language = spark.table("steam.bridge_game_language")
bridge_game_tags = spark.table("steam.bridge_game_tags")
bridge_game_platform = spark.table("steam.bridge_game_platform")
dim_platform = spark.table("steam.dim_platform")

# QUESTIONS

In [0]:
# Q1 — Publishers with the most games
top_publishers = (
    bridge_game_publisher
        .groupBy("publisher")
        .agg(F.countDistinct("appid").alias("games_count"))
        .orderBy(F.desc("games_count"))
        .limit(15)
)

display(top_publishers)

publisher,games_count
Big Fish Games,424
8floor,244
Sega,187
Square Enix,151
Strategy First,151
Thq Nordic,144
Choice Of Games,140
Sekai Project,136
Plug In Digital,136
Hh-games,133


Databricks visualization. Run in Databricks to view.

In [0]:
# Q2 — Top 3 best rated games with inline images (fixed)
# DO NOT: from IPython.display import display
from IPython.display import HTML  # optional, not strictly needed

MIN_REVIEWS = 1000

top3 = (
    fact_games
        .select(
            "appid", "name", "release_year", "header_image",
            F.col("positive").cast("long").alias("pos"),
            F.col("negative").cast("long").alias("neg")
        )
        .withColumn("total_reviews", F.col("pos") + F.col("neg"))
        .withColumn("positive_rate", F.when(F.col("total_reviews") > 0, F.col("pos") / F.col("total_reviews")))
        .filter(F.col("total_reviews") >= MIN_REVIEWS)
        .orderBy(F.desc("positive_rate"), F.desc("total_reviews"))
        .limit(3)
        .withColumn("positive_rate_pct", F.round(F.col("positive_rate") * 100, 2))
        .select("name", "release_year", "total_reviews", "positive_rate_pct", "header_image")
)

rows = top3.collect()

cards = "".join(f"""
<div style='text-align:center;width:220px;'>
  <img src='{r.header_image}' width='200' style='border-radius:10px;box-shadow:0 2px 6px rgba(0,0,0,0.3);'><br>
  <b>{r.name}</b><br>
  <span style='color:gray;'>{r.release_year}</span><br>
  <span style='font-size:13px;'>{r.positive_rate_pct}% positive • {r.total_reviews} reviews</span>
</div>
""" for r in rows)

displayHTML(f"<div style='display:flex;gap:30px;'>{cards}</div>")

Aventura Copilului Albastru și Urât 
 2021 
 99.37% positive • 2217 reviews 
 

 
 
 Aseprite 
 2016 
 99.33% positive • 11903 reviews 
 

 
 
 Patrick's Parabox 
 2022 
 99.27% positive • 1500 reviews

In [0]:
# Q3 — Number of releases per year
from pyspark.sql import functions as F

releases_by_year = (
    fact_games
      .select(F.col("release_year").cast("int").alias("year"), "appid")
      .where(F.col("year").isNotNull())
      .groupBy("year")
      .agg(F.countDistinct("appid").alias("games_released"))
      .orderBy("year")
)

display(releases_by_year)

year,games_released
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


Databricks visualization. Run in Databricks to view.

In [0]:
# Q4 — Pricing
# 1) Price distribution
price_counts = (
    fact_games
        .filter(
            F.col("price_eur").isNotNull() &
            (F.col("price_eur") > 0) &
            (F.col("price_eur") <= 300) # as there is an outlier at 1000 euros
        )
        .groupBy(F.round(F.col("price_eur"), 2).alias("price_eur"))
        .agg(F.count("*").alias("nb_games"))
        .orderBy("price_eur")
)
display(price_counts)

price_eur,nb_games
0.5,1
0.9,34
0.98,2
0.99,5441
1.0,17
1.29,4
1.33,1
1.36,1
1.4,1
1.42,1


Databricks visualization. Run in Databricks to view.

In [0]:
 # Q4 — Pricing
 # 2) Discount statistics
from pyspark.sql.window import Window

# Optional: make nulls explicit so they appear as a third category
df = fact_games.withColumn(
    "discount_flag",
    F.when(F.col("is_discounted").isNull(), F.lit("Unknown"))
     .otherwise(F.col("is_discounted").cast("string"))
)

discount_stats = (
    df.groupBy("discount_flag")
      .agg(F.count("*").alias("nb_games"))
      # total across all rows of this small aggregated DF
      .withColumn("total", F.sum("nb_games").over(Window.partitionBy(F.lit(1))))
      .withColumn("pct", F.round(F.col("nb_games") / F.col("total") * 100, 2))
      .select(F.col("discount_flag").alias("is_discounted"), "nb_games", "pct")
      .orderBy(F.desc("nb_games"))
)

display(discount_stats)

is_discounted,nb_games,pct
false,53173,95.48
true,2518,4.52


Databricks visualization. Run in Databricks to view.

In [0]:
# Q5 — Most represented languages
popular_languages = (
    bridge_game_language
        .filter(F.col("language").isNotNull() & (F.trim(F.col("language")) != ""))
        .groupBy(F.col("language"))
        .agg(F.count("*").alias("nb_games"))
        .orderBy(F.desc("nb_games"))
        .limit(10)
)

display(popular_languages)

language,nb_games
English,55122
German,14021
French,13428
Russian,12925
Spanish,12843
Simplified Chinese,12782
Japanese,10371
Italian,9306
Portuguese,8471
Korean,6601


Databricks visualization. Run in Databricks to view.

In [0]:
# Q6 — Games restricted to 16+
restricted_games = (
    fact_games
        .filter(F.col("required_age_clean") >= 16)
        .groupBy("required_age_clean")
        .agg(F.countDistinct("appid").alias("nb_games"))
        .orderBy("required_age_clean")
)

display(restricted_games)

required_age_clean,nb_games
16,38
17,38
18,223
20,1
21,1


In [0]:
# Q6 — Games restricted to 16+ that have adult/violent themes
from pyspark.sql import functions as F

# Define adult/violent theme matcher (case-insensitive)
adult_regex = r"(?i)\b(violence|violent|gore|blood|nudity|sexual|sex|adult|mature|erotic)\b"

# Keep only 16+ games
g16 = fact_games.filter(F.col("required_age_clean") >= 16).select("appid", "required_age_clean")

# Join to tags and keep only adult/violent ones
g16_adult = (
    g16.alias("f")
      .join(bridge_game_tags.alias("t"), "appid", "inner")
      .filter(F.col("t.tag").isNotNull() & (F.trim(F.col("t.tag")) != ""))
      .filter(F.col("t.tag").rlike(adult_regex))
      .withColumn("tag_norm", F.initcap(F.trim(F.col("t.tag"))))
      .select("appid", "required_age_clean", "tag_norm")
      .distinct()
)

# 1) Breakdown by tag within 16+ games
q6_by_tag = (
    g16_adult
      .groupBy("tag_norm", "required_age_clean")
      .agg(F.countDistinct("appid").alias("nb_games"))
      .orderBy(F.desc("nb_games"))
)

display(q6_by_tag)

# 2) Overall: how many 16+ games have ANY adult/violent tag (+ percentage of all 16+)
total_16 = g16.agg(F.countDistinct("appid").alias("total_16")).collect()[0]["total_16"]

q6_any = (
    g16_adult
      .groupBy("required_age_clean")
      .agg(F.countDistinct("appid").alias("nb_games_adult"))
      .withColumn("pct_of_16", F.round(F.col("nb_games_adult") / F.lit(total_16) * 100, 2))
      .orderBy("required_age_clean")
)

display(q6_any)


tag_norm,required_age_clean,nb_games
Violent,18,96
Gore,18,96
Nudity,18,89
Sexual Content,18,77
Mature,18,44
Blood,18,23
Gore,17,18
Violent,16,13
Gore,16,11
Nudity,17,9


Databricks visualization. Run in Databricks to view.

required_age_clean,nb_games_adult,pct_of_16
16,19,6.31
17,25,8.31
18,177,58.8
20,1,0.33
21,1,0.33


In [0]:
# Q7 — Most represented genres
top_genres = (
    bridge_game_genre
        .filter(F.col("genre").isNotNull() & (F.trim(F.col("genre")) != ""))
        .groupBy(F.col("genre"))
        .agg(F.countDistinct("appid").alias("nb_games"))
        .orderBy(F.desc("nb_games"))
        .limit(10)
)

display(top_genres)

genre,nb_games
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

# Q8 — Genres with best weighted positive rate (%)
MIN_REVIEWS_PER_GAME  = 100     # ignore games with too few reviews
MIN_REVIEWS_PER_GENRE = 1000    # require enough total reviews per genre

gr = (
    fact_games
        .select("appid", "positive", "negative")
        .withColumn("pos", F.col("positive").cast("long"))
        .withColumn("neg", F.col("negative").cast("long"))
        .withColumn("tot", F.col("pos") + F.col("neg"))
        .filter(F.col("tot") >= MIN_REVIEWS_PER_GAME)
        .join(bridge_game_genre.select("appid", "genre"), "appid", "inner")
        .groupBy("genre")
        .agg(
            F.sum("pos").alias("sum_pos"),
            F.sum("neg").alias("sum_neg"),
            F.countDistinct("appid").alias("n_games")
        )
        .withColumn("total_reviews", F.col("sum_pos") + F.col("sum_neg"))
        .filter(F.col("total_reviews") >= MIN_REVIEWS_PER_GENRE)
        .withColumn("positive_rate_weighted_pct",
                    F.round(F.col("sum_pos") / F.col("total_reviews") * 100, 2))
        .select("genre", "positive_rate_weighted_pct", "total_reviews", "n_games")
)

# --- Top 10 genres (best rating)
top_genres = gr.orderBy(F.desc("positive_rate_weighted_pct"), F.desc("total_reviews")).limit(10)
display(top_genres)

# --- Bottom 10 genres (lowest rating)
bottom_genres = gr.orderBy(F.asc("positive_rate_weighted_pct"), F.desc("total_reviews")).limit(10)
display(bottom_genres)

genre,positive_rate_weighted_pct,total_reviews,n_games
Photo Editing,97.74,590014,20
Animation & Modeling,96.55,711283,85
Design & Illustration,96.4,693687,92
Utilities,94.8,771700,155
Game Development,90.36,28137,32
Audio Production,88.97,75158,43
Indie,88.73,36133776,10607
Video Production,87.9,123853,64
Web Publishing,87.22,37946,34
Casual,87.12,11217985,4735


Databricks visualization. Run in Databricks to view.

genre,positive_rate_weighted_pct,total_reviews,n_games
Movie,52.31,3154,1
Massively Multiplayer,73.11,10897841,768
Nudity,74.27,9759,22
Violent,79.83,49174,45
Sports,80.44,3631459,760
Software Training,80.64,25438,34
Free to Play,81.42,22948295,1890
Education,81.69,22171,33
Sexual Content,82.14,6958,20
Early Access,82.45,5176356,1373


Databricks visualization. Run in Databricks to view.

In [0]:
# Q9 — Publishers’ favorite genres

TOP_PUBLISHERS = 15
TOP_K = 3

top_publishers = (
    bridge_game_publisher
        .filter(F.col("publisher").isNotNull() & (F.trim(F.col("publisher")) != ""))
        .groupBy("publisher")
        .agg(F.countDistinct("appid").alias("games_count"))
        .orderBy(F.desc("games_count"))
        .limit(TOP_PUBLISHERS)
)

pub_genre = (
    bridge_game_publisher.select("appid", "publisher")
    .join(bridge_game_genre.select("appid", "genre"), "appid", "inner")
    .filter(F.col("genre").isNotNull() & (F.trim(F.col("genre")) != ""))
    .groupBy("publisher", "genre")
    .agg(F.countDistinct("appid").alias("n_games"))
)

# keep only top publishers, compute % of each genre within publisher
pub_genre_pct = (
    pub_genre.join(top_publishers.select("publisher", "games_count"), "publisher", "inner")
             .withColumn("pct_of_catalog", F.round(F.col("n_games") / F.col("games_count") * 100, 2))
)

# rank genres within each publisher and keep top-K
w = Window.partitionBy("publisher").orderBy(F.desc("n_games"), F.asc("genre"))
topk_per_publisher = (
    pub_genre_pct
        .withColumn("rnk", F.dense_rank().over(w))
        .filter(F.col("rnk") <= TOP_K)
        .select("publisher", "genre", "n_games", "pct_of_catalog", "rnk")
        .orderBy("publisher", "rnk", F.desc("n_games"))
)
# Preserve the publisher order (by total games) for visualization
ordered_publishers = (
    top_publishers
        .orderBy(F.desc("games_count"))
        .withColumn("publisher_order", F.monotonically_increasing_id())
)

topk_sorted = (
    topk_per_publisher
        .join(ordered_publishers.select("publisher", "publisher_order"), "publisher")
        .orderBy("publisher_order", "rnk")
)

display(topk_sorted)

publisher,genre,n_games,pct_of_catalog,rnk,publisher_order
Big Fish Games,Casual,420,99.06,1,0
Big Fish Games,Adventure,394,92.92,2,0
Big Fish Games,Indie,7,1.65,3,0
8floor,Casual,244,100.0,1,1
8floor,Strategy,39,15.98,2,1
8floor,Adventure,14,5.74,3,1
Sega,Action,88,47.06,1,2
Sega,Strategy,53,28.34,2,2
Sega,Adventure,33,17.65,3,2
Strategy First,Strategy,52,34.44,1,3


Databricks visualization. Run in Databricks to view.

In [0]:
# Q10 — Most lucrative genres (estimated)
# revenue ≈ price_after_discount (€) * owners_min (conservative lower bound)

MAX_PRICE = 300  # cap to ignore outliers (e.g., 1000 €)
TOP_K = 15

genre_revenue = (
    fact_games.alias("f")
      .select(
          "appid",
          F.col("price_after_discount").cast("double").alias("price"),
          F.col("owners_min").cast("long").alias("owners_min")
      )
      .filter(
          F.col("price").isNotNull() & (F.col("price") > 0) & (F.col("price") <= MAX_PRICE) &
          F.col("owners_min").isNotNull() & (F.col("owners_min") > 0)
      )
      .join(bridge_game_genre.select("appid", "genre"), "appid", "inner")
      .groupBy("genre")
      .agg(
          F.round(F.sum(F.col("price") * F.col("owners_min")), 2).alias("estimated_revenue_eur"),
          F.countDistinct("appid").alias("n_games"),
          F.round(F.avg("price"), 2).alias("avg_price_eur")
      )
      .orderBy(F.desc("estimated_revenue_eur"))
      .limit(TOP_K)
)

display(genre_revenue)

genre,estimated_revenue_eur,n_games,avg_price_eur
Action,3.59305337E10,6114,12.25
Adventure,2.26198649E10,5782,11.52
Indie,1.91354431E10,9185,9.41
RPG,1.66760684E10,3092,13.32
Strategy,1.23626666E10,3265,12.57
Simulation,1.1422447E10,2748,13.86
Casual,4.4767689E9,4180,7.11
Massively Multiplayer,3.6932486E9,234,13.67
Early Access,3.1471007E9,982,13.45
Sports,1.816062E9,488,15.62


Databricks visualization. Run in Databricks to view.

In [0]:
# Q11 — On which platforms are most games available?
total_games_df = bridge_game_platform.select(F.countDistinct("appid").alias("total_games"))

platform_stats = (
    bridge_game_platform
        .groupBy("platform")
        .agg(F.countDistinct("appid").alias("n_games"))
        .crossJoin(total_games_df)
        .withColumn("pct_games", F.round(F.col("n_games") / F.col("total_games") * 100, 2))
        .select(
            F.col("platform").alias("Platform"),
            F.col("n_games").alias("Number of Games"),
            F.col("pct_games").alias("Share of Total (%)")
        )
        .orderBy(F.desc("Number of Games"))
)

display(platform_stats)

Platform,Number of Games,Share of Total (%)
Windows,55676,99.97
Mac,12770,22.93
Linux,8458,15.19


Databricks visualization. Run in Databricks to view.

In [0]:
# Q12 — Platform distribution per genre (normalized to 100%)
MIN_GENRE_SIZE = 200   # keep only well-represented genres
TOP_GENRES = 20        # limit for readability

# per-genre × platform counts
gp = (
    bridge_game_genre.select("appid", "genre")
    .join(bridge_game_platform.select("appid", "platform"), "appid", "inner")
    .join(dim_platform, F.col("platform") == F.col("platform_name"), "inner")
    .groupBy("genre", "platform_name")
    .agg(F.countDistinct("appid").alias("n_games_platform"))
)

# total per genre
genre_totals = (
    bridge_game_genre
    .groupBy("genre")
    .agg(F.countDistinct("appid").alias("n_games_genre"))
)

# long format with share within genre (adds up to 100%)
genre_platform_share = (
    gp.join(genre_totals, "genre")
      .withColumn("share_within_genre", F.round(F.col("n_games_platform") / F.col("n_games_genre") * 100, 2))
)

# keep only biggest genres
top_genres = genre_totals.orderBy(F.desc("n_games_genre")).limit(TOP_GENRES).select("genre")
q12_long = genre_platform_share.join(top_genres, "genre", "inner")

display(q12_long)

genre,platform_name,n_games_platform,n_games_genre,share_within_genre
RPG,Windows,9533,9534,99.99
Sports,Windows,2665,2666,99.96
Violent,Windows,168,168,100.0
Simulation,Windows,10832,10836,99.96
Free to Play,Linux,474,3393,13.97
Utilities,Windows,681,682,99.85
Indie,Linux,6978,39681,17.59
Racing,Windows,2154,2155,99.95
Software Training,Windows,164,164,100.0
Video Production,Linux,6,247,2.43


Databricks visualization. Run in Databricks to view.